# Imports, API Credentials & Configuration

In [1]:
import hashlib
import time
import numpy as np
import pandas as pd
import requests

API_KEY = "3a17981485d39519a9e7ef201a9d7ac27d02657d"  # Replace with your actual
API_SECRET = "ea7252fdf927e207f42eb843a056f48257de6986"  # Replace with your
CONTEST_ID = 675515  # Group contest ID
TYPE = "contest"  # Options: "contest", "sheet", "offline_contest"


# Helper function to generate signed parameters for Codeforces API
def get_signed_params(method, extra_params):
    rand_prefix = "123456"
    params = extra_params.copy()
    params["apiKey"] = API_KEY
    params["time"] = str(int(time.time()))

    sorted_params = sorted(params.items())
    param_str = "&".join([f"{k}={v}" for k, v in sorted_params])

    sig_base = f"{rand_prefix}/{method}?{param_str}#{API_SECRET}"
    api_sig = rand_prefix + hashlib.sha512(sig_base.encode("utf-8")).hexdigest()
    params["apiSig"] = api_sig

    return params

# Fetch Standings & Create Base DataFrame

In [2]:
# Fetch contest standings
standings_params = get_signed_params(
    "contest.standings",
    {"contestId": str(CONTEST_ID), "showUnofficial": "true"},
)

resp = requests.get(
    "https://codeforces.com/api/contest.standings", params=standings_params
)
print("Status Code:", resp.status_code)

data = resp.json()
result = data["result"]
contest_name = result["contest"]["name"]
problems = [p["index"] for p in result["problems"]]

standings_list = []

for row in result["rows"]:
    handle = row["party"]["members"][0]["handle"]
    player_data = {
        "Rank": row["rank"],
        "Handle": handle,
        "Solved": int(row["points"]),
        "Penalty": row["penalty"],
    }

    # Format problem statuses (+ / - / .)
    for prob, res in zip(result["problems"], row["problemResults"]):
        prob_index = prob["index"]
        solved = res["points"] > 0
        attempts = res["rejectedAttemptCount"]

        if solved:
            player_data[prob_index] = f"+{attempts}" if attempts > 0 else "+"
        elif attempts > 0:
            player_data[prob_index] = f"-{attempts}"
        else:
            player_data[prob_index] = "."

    standings_list.append(player_data)

# Create DataFrame
df = pd.DataFrame(standings_list)

# Remove unranked participants (Rank == 0)
df = df[df["Rank"] != 0].reset_index(drop=True)
total_problems = len(problems)

print(f"Loaded {len(df)} participants across {total_problems} problems.")

Status Code: 200
Loaded 15 participants across 7 problems.


# Fetch First Solves

In [3]:
# Fetch submissions to find the first solver per problem (Only for contest / offline_contest)
if TYPE in ["contest", "offline_contest"]:
    # Initialize Fastest_Solve column only for contest and offline_contest
    df["Fastest_Solve"] = 0

    status_params = get_signed_params(
        "contest.status", {"contestId": str(CONTEST_ID)}
    )
    status_resp = requests.get(
        "https://codeforces.com/api/contest.status", params=status_params
    )

    if status_resp.status_code == 200 and status_resp.json()["status"] == "OK":
        submissions = status_resp.json()["result"]

        first_solvers = {}
        #FIRST_SOLVE_BONUS = 50

        for prob_idx in problems:
            # Filter accepted submissions for this problem
            ac_submissions = [
                sub
                for sub in submissions
                if sub.get("verdict") == "OK"
                and sub["problem"]["index"] == prob_idx
            ]

            if ac_submissions:
                # Find earliest submission by creation timestamp
                first_sub = min(
                    ac_submissions, key=lambda x: x["creationTimeSeconds"]
                )
                solver_handle = first_sub["author"]["members"][0]["handle"]
                first_solvers[prob_idx] = solver_handle

                # Apply bonus points and increment the fastest solve counter
                mask = df["Handle"] == solver_handle
                if mask.any():
                    #df.loc[mask, "Score"] += FIRST_SOLVE_BONUS
                    df.loc[mask, "Fastest_Solve"] += 1

        #print("First Solvers per Problem:", first_solvers)

# Calculate Scores

In [4]:
# Calculate base scores according to TYPE
if TYPE == "sheet":
    solved_ratio = df["Solved"] / total_problems
    conditions = [
        (solved_ratio >= 0.5) & (solved_ratio < 0.75),
        (solved_ratio >= 0.75) & (solved_ratio < 1.0),
        (solved_ratio == 1.0),
    ]
    scores = [150, 300, 400]
    df["Score"] = np.select(conditions, scores, default=0)

elif TYPE == "contest":
    df["Score"] = df["Solved"] * 50 + df["Fastest_Solve"] * 20
    # Top 3 Rank Bonuses
    if len(df) >= 1:
        df.iloc[0, df.columns.get_loc("Score")] += 300
    if len(df) >= 2:
        df.iloc[1, df.columns.get_loc("Score")] += 200
    if len(df) >= 3:
        df.iloc[2, df.columns.get_loc("Score")] += 100

elif TYPE == "offline_contest":
    df["Score"] = df["Solved"] * 70 + df["Fastest_Solve"] * 50
    # Top 3 Rank Bonuses
    if len(df) >= 1:
        df.iloc[0, df.columns.get_loc("Score")] += 300
    if len(df) >= 2:
        df.iloc[1, df.columns.get_loc("Score")] += 200
    if len(df) >= 3:
        df.iloc[2, df.columns.get_loc("Score")] += 100

# Export to Excel & Display Results

In [5]:
# Reorder columns to place Fastest_Solve cleanly before Penalty if present
if "Fastest_Solve" in df.columns:
    base_cols = ["Rank", "Handle", "Solved", "Fastest_Solve", "Penalty", "Score"]
else:
    base_cols = ["Rank", "Handle", "Solved", "Penalty", "Score"]

# Export DataFrame to CSV
df.to_csv("sheet1_standings.csv", index=False)

print(f"Sheet Name: {contest_name}")
df[base_cols + problems[:]].head()

Sheet Name: Contest #2


,Rank,Handle,Solved,Fastest_Solve,Penalty,Score,A,B,C,D,E,F,H
0,1,HosSam_Abu_elez,7,0,174,650,+,+,+,+,+,+,+3
1,2,Yousef_maaty20,6,1,156,520,+,+,+,+,+,+,.
2,3,Hend_22,6,4,177,480,+,+,+,+,+1,+,.
3,4,_7azzzemm_,6,0,193,300,+,+,+,+,+,+,-4
4,5,Fa7L,6,1,209,320,+,+,+,+,+,+,.
